In [4]:
#Let's do load the csv file

import pandas as pd
import numpy as np


df_train = pd.read_csv("training_yoga1_keypoints.csv")
x_training = df_train.drop(columns=['label']).to_numpy()  # shape (num_samples, 99)
y_training = df_train['label'].to_numpy()                # shape (num_samples,)

df_val = pd.read_csv("validation_yoga1_keypoints.csv")
x_validation = df_val.drop(columns=['label']).to_numpy()
y_validation = df_val['label'].to_numpy()

df_test = pd.read_csv("test_yoga1_keypoints.csv")
x_test = df_test.drop(columns=['label']).to_numpy()
y_test = df_test['label'].to_numpy()

In [5]:
print(x_validation)

[[ 0.39865306  0.32344198 -0.42548376 ...  0.71660268  0.57827866
   0.32334566]
 [ 0.31903014  0.4237856  -0.42195144 ...  0.88069779  0.73749006
   0.25335595]
 [ 0.29657516  0.66703618 -0.64195538 ...  0.82001114  0.80451971
  -0.46697673]
 ...
 [ 0.44415075  0.58295166 -0.30628198 ...  0.71366137  0.5874297
   0.80619586]
 [ 0.15924285  0.40728283 -0.389595   ...  0.93443173  0.92006761
   0.06681839]
 [ 0.43507791  0.05512149 -0.99150664 ...  0.53051764  0.58052248
   1.44121099]]


In [6]:
print("X shape:", x_training.shape)       # should be (4196, 99)
print("Y shape:", y_training.shape)       # should be (4196,)
print("X val shape:", x_validation.shape)
print("Y val shape:", y_validation.shape)

X shape: (4196, 99)
Y shape: (4196,)
X val shape: (901, 99)
Y val shape: (901,)


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

num_keypoints = 33
feature_per_keypoints = 3 # potentially only have xyz
input_dim = feature_per_keypoints * num_keypoints
num_classes = 107

model = models.Sequential([
    layers.Input(shape=(99,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss = 'sparse_categorical_crossentropy',
    metrics=['accuracy']
)



early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(x_training, y_training, validation_data=(x_validation, y_validation),
          epochs=10, callbacks=[early_stop])

Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0319 - loss: 4.5620 - val_accuracy: 0.0688 - val_loss: 4.3837
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 798us/step - accuracy: 0.0934 - loss: 4.0705 - val_accuracy: 0.1110 - val_loss: 3.9549
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 827us/step - accuracy: 0.1408 - loss: 3.7347 - val_accuracy: 0.1410 - val_loss: 3.7852
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 843us/step - accuracy: 0.1745 - loss: 3.5008 - val_accuracy: 0.1709 - val_loss: 3.6759
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 908us/step - accuracy: 0.2071 - loss: 3.3465 - val_accuracy: 0.1709 - val_loss: 3.6162
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 825us/step - accuracy: 0.2300 - loss: 3.2251 - val_accuracy: 0.1964 - val_loss: 3.5052
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 834us/step - accuracy: 0.2593 - loss: 3.1006 - val_accuracy: 0.2209 - val_loss: 3.4864
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 776us/step - accuracy: 0.2834 - loss: 3.0005 - va

In [8]:
model.evaluate(x_test,y_test)

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 649us/step - accuracy: 0.2433 - loss: 3.3353


[3.3352794647216797, 0.2433035671710968]

In [10]:
from sklearn.metrics import f1_score
import numpy as np

# Predict on x_test
y_pred_prob = model.predict(x_test)
y_pred = np.argmax(y_pred_prob, axis=1)

# True labels
y_true = np.argmax(y_test, axis=1) if y_test.ndim > 1 else y_test

# Macro F1 (treats all classes equally)
f1_macro = f1_score(y_true, y_pred, average='macro')

# Weighted F1 (weighted by support)
f1_weighted = f1_score(y_true, y_pred, average='weighted')

print("Macro F1:", f1_macro)
print("Weighted F1:", f1_weighted)

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 469us/step
Macro F1: 0.2158922509915153
Weighted F1: 0.23483872317775015


I want to turn this data I read from csv into a data for my nueral network.